# ALMA spectral energy distribution: simulate and recover

[Colab Link](https://colab.research.google.com/github/casangi/astroviper/blob/main/docs/distributed_applications_tutorials/simulation/alma_sed_simulation.ipynb)

The radio to millimetre spectral energy distribution (SED) of a local starburst nucleus, built from the four
components of Yoon et al. (2026, ApJ 1004, 157): thermal dust, free-free, synchrotron and anomalous microwave
emission (AME). Their sum is observed with the ALMA 12 m array as a single field in every band, with the noise of the
ALMA Sensitivity Calculator, and imaged with the cube imager. The test is whether the imaged spectrum recovers the
simulated total.

---
## API

In [ ]:
from astroviper.distributed_applications.simulation import simulate_processing_set

simulate_processing_set?

## Install AstroVIPER

In [ ]:
import os
from importlib.metadata import version

try:
    import astroviper  # noqa: F401

    print("Using astroviper version", version("astroviper"))
except ImportError:
    os.system("pip install --upgrade astroviper")
    import astroviper  # noqa: F401

    print("Installed astroviper version", version("astroviper"))

In [ ]:
%matplotlib inline
import importlib.resources

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
from astropy import constants as const
from astropy import units as u
from astropy.cosmology import FlatLambdaCDM
from xradio.image import load_image
from xradio.measurement_set import open_processing_set

from astroviper.distributed_applications.imaging import image_cube_single_field
from astroviper.utils.beam_models import airy_disk_model
from astroviper.utils.telescope_layout import read_telescope_layout

In [ ]:
from toolviper.dask.client import local_client

quiet = {"log_level": "WARNING"}
viper_client = local_client(
    cores=4,
    memory_limit="4GB",
    log_params=quiet,
    worker_log_params=quiet,
)
viper_client

## SED model

The source is at $z = 0.01$ ($D_L = 43.2$ Mpc for the paper's $H_0 = 70$ km s$^{-1}$ Mpc$^{-1}$, $\Omega_M = 0.3$)
and taken as unresolved (spatially integrated, as in the paper), so the SED is a point source at the phase centre. A
rest-frame luminosity density $L_\nu$ is observed as $S_\nu(\nu) = (1+z)\,L_\nu\big((1+z)\nu\big) / 4\pi D_L^2$, and the simulated SED is the sum of four components:

- **Thermal dust** (Section 3.4): CIGALE's `casey2012` model, a general-opacity modified blackbody
  ($\lambda_0 = 200\ \mu$m, $\beta = 1.6$) plus a mid-infrared power law ($\alpha = 2$), normalised to the dust
  luminosity $L_\mathrm{dust} = 10^{11}\,L_\odot$, with $T_{d,*} = 30$ K. The CMB heats the dust,
  $T_d^{4+\beta}(z) = T_{d,*}^{4+\beta} + T_\mathrm{CMB}^{4+\beta}(z)$ (Eq. 6), and lowers its contrast by
  $1 - B_\nu(T_\mathrm{CMB}) / B_\nu(T_d)$.
- **Synchrotron and free-free** (Section 3.3): the radio-IR correlation $q = 2.64$ (Eq. 4) gives
  $L_{1.4\,\mathrm{GHz}} = L_\mathrm{dust} / (3.75 \times 10^{12}\ \mathrm{Hz} \cdot 10^{q})$, of which 90 % is
  synchrotron ($\propto \nu^{-0.8}$) and 10 % free-free ($\propto \nu^{-0.1}$). The synchrotron dimming factor
  (Eq. 3) is taken as 1 at this redshift.
- **AME** (Section 3.5): the SpDust emissivity $j_\nu / n_H$ of a photodissociation region, with the radiation
  field raised by the CMB (Eq. 10), from a region of diameter $l = 100$ pc and column density
  $N_H = 10^{25}$ cm$^{-2}$ (Eq. 7): $L_\nu = 4\pi\,(j_\nu / n_H)\,N_H\,\pi l^2 / 4$. This $N_H$ is the
  column the paper chose deliberately high, an extreme environment for strong AME (Section 5.1), and the paper keeps
  the AME out of its galaxy SEDs (Section 2.3). The emissivity is the PDR column of `astroviper/data/simulation/spdust_emissivity.csv`, written by
  `spdust_emissivity_generator.py` next to it with SpyDust.

In [ ]:
h, k, c = const.h.value, const.k_B.value, const.c.value
Z = 0.01
D_L = FlatLambdaCDM(H0=70, Om0=0.3).luminosity_distance(Z).to(u.m).value
L_DUST = 1e11 * const.L_sun.value  # W
T_STAR, BETA, ALPHA = 30.0, 1.6, 2.0
T_CMB = 2.725 * (1 + Z)
T_D = (T_STAR ** (4 + BETA) + T_CMB ** (4 + BETA)) ** (1 / (4 + BETA))
L_1P4 = L_DUST / (3.75e12 * 10**2.64)  # W/Hz
L_AME, N_H = (100 * u.pc).to(u.cm).value, 1e25
spdust = pd.read_csv(
    importlib.resources.files("astroviper")
    / "data"
    / "simulation"
    / "spdust_emissivity.csv",
    comment="#",
    names=["nu_GHz", "CNM", "WNM", "PDR"],
)


def planck(nu, t):
    return nu**3 / np.expm1(h * nu / (k * t))


def casey(nu, t):
    """Modified blackbody and power law of CIGALE's casey2012, as L_nu shapes."""
    lam, lam_0 = c / nu, 200e-6
    lam_c = 0.75e-6 / (
        (26.68 + 6.246 * ALPHA) ** -2 + (1.905e-4 + 7.243e-5 * ALPHA) * t
    )
    mbb = -np.expm1(-((lam_0 / lam) ** BETA)) * planck(nu, t)
    n_pl = -np.expm1(-((lam_0 / lam_c) ** BETA)) * planck(c / lam_c, t)
    return mbb, n_pl * (lam / lam_c) ** ALPHA * np.exp(-((lam / lam_c) ** 2))


nu_ir = np.geomspace(c / 1e-3, c / 1e-6, 4000)  # 1 um to 1 mm, as CIGALE
dust_norm = np.trapezoid(np.sum(casey(nu_ir, T_D), axis=0), nu_ir)


def dust(nu):
    mbb, power_law = casey(nu, T_D)
    contrast = 1 - planck(nu, T_CMB) / planck(nu, T_D)
    heating = (T_D / T_STAR) ** (4 + BETA)
    return L_DUST * heating * (mbb * contrast + power_law) / dust_norm


def ame(nu):
    log_j = np.interp(
        np.log(nu / 1e9), np.log(spdust.nu_GHz), np.log(np.maximum(spdust.PDR, 1e-300))
    )
    return (
        4 * np.pi * np.exp(log_j) * N_H * np.pi * L_AME**2 / 4 * 1e-30
    )  # Jy cm^2 to W/Hz


LUMINOSITY = {
    "dust": dust,
    "free-free": lambda nu: 0.1 * L_1P4 * (nu / 1.4e9) ** -0.1,
    "synchrotron": lambda nu: 0.9 * L_1P4 * (nu / 1.4e9) ** -0.8,
    "AME": ame,
}


def flux_density(component, nu):
    """Observed flux density (Jy) of one component at the observed frequency nu (Hz)."""
    return (1 + Z) * LUMINOSITY[component]((1 + Z) * nu) / (4 * np.pi * D_L**2) * 1e26


def total_flux_density(nu):
    return sum(flux_density(component, nu) for component in LUMINOSITY)

## ALMA bands and noise

One continuum tuning per band: four channels over 7.5 GHz (15 GHz in the double-sideband Bands 9 and 10), 1 h on
source with the 43 antennas of configuration C-3 (`alma.cycle8.3`), at (RA, Dec) = (0°, 0°) as in the paper's
Table 2. $T_\mathrm{sys}$ and the 1 h continuum sensitivity $\sigma$ are those of the ALMA Sensitivity Calculator
(automatic water vapour column, read on 18 Nov 2026). Run with the same settings at the paper's Table 2
frequencies, the calculator gives 10.6, 11.8, 20.1 and 25.9 µJy at 93, 150, 250 and 350 GHz, against 11, 12, 20 and
26 µJy in Table 2.

The simulator's noise model follows CASA's `tsys-manual` formula without its opacity term (`tau` is accepted but not
used), so the calculator's $T_\mathrm{sys}$, which includes the atmosphere and the CMB, is passed as `t_receiver` with
`t_atmos = t_cmb = 0`, and `ant_efficiency` is the value for which the radiometer equation, with correlator and
quantization efficiencies of 0.88 and 0.96, gives $\sigma$. In every band it equals
$0.649\,\exp[-(4\pi \cdot 25\ \mu\mathrm{m} / \lambda)^2]$ to 0.01 %, a Ruze law for a 25 µm surface.

In [ ]:
N_ANT, T_OBS, N_CHAN = 43, 3600.0, 4
ETA_C, ETA_Q = 0.88, 0.96  # correlator and quantization efficiencies
bands = pd.DataFrame(
    [  # band, centre (GHz), bandwidth (GHz), water vapour (mm), T_sys (K), sigma in 1 h (uJy)
        (1, 43.0, 7.5, 5.186, 59.024, 8.434),
        (2, 75.0, 7.5, 5.186, 81.693, 11.722),
        (3, 97.5, 7.5, 5.186, 74.900, 10.793),
        (4, 145.0, 7.5, 2.748, 80.678, 11.774),
        (5, 203.0, 7.5, 1.796, 104.023, 15.521),
        (6, 233.0, 7.5, 1.796, 102.719, 15.548),
        (7, 343.5, 7.5, 1.262, 182.991, 29.706),
        (8, 405.0, 7.5, 0.913, 313.715, 53.567),
        (9, 679.0, 15.0, 0.658, 1154.999, 193.233),
        (10, 870.0, 15.0, 0.658, 1992.442, 461.303),
    ],
    columns=["band", "freq_GHz", "bandwidth_GHz", "pwv_mm", "t_sys_K", "sigma_uJy"],
).set_index("band")
radiometer = np.sqrt(N_ANT * (N_ANT - 1) * 2 * bands.bandwidth_GHz * 1e9 * T_OBS)
bands["ant_efficiency"] = (
    2
    * k
    * bands.t_sys_K
    / (ETA_C * ETA_Q * np.pi * 6.0**2 * bands.sigma_uJy * 1e-32 * radiometer)
)
bands.round(3)

## Simulate the total in each band

`simulate_processing_set` writes one processing set per band, 10 in all: XX and YY, 60 integrations of 60 s
around transit, an Airy-disk primary beam, the total flux density in every channel and a noise seed of its own.
The report of the Band 3 run is shown below the cell.

In [ ]:
antenna_xds = read_telescope_layout("alma.cycle8.3")
phase_center = np.array([[0.0, 0.0]])  # RA, Dec (rad)
time_params = {
    "time_start": "2026-10-01T03:21:00.000",
    "time_delta": 60.0,
    "n_samples": 60,
}  # transit at 03:53 UTC


def store(band, kind):
    return f"sed_B{band}.{kind}.zarr"


def channel_frequencies(band):
    width = bands.loc[band, "bandwidth_GHz"] / N_CHAN * 1e9
    centre = bands.loc[band, "freq_GHz"] * 1e9
    return centre + (np.arange(N_CHAN) - (N_CHAN - 1) / 2) * width, width


for band, row in bands.iterrows():
    freqs, width = channel_frequencies(band)
    simulate_processing_set(
        ps_store=store(band, "ps"),
        antenna_xds=antenna_xds,
        time_params=time_params,
        frequency_params={
            "freq_start": freqs[0],
            "freq_delta": width,
            "n_channels": N_CHAN,
            "channel_width": width,
            "spectral_window_name": f"ALMA_B{band}",
        },
        polarization=["XX", "YY"],
        phase_center_ra_dec=phase_center,
        beam_models=[airy_disk_model("alma")],
        beam_model_map=np.zeros(N_ANT, dtype=int),
        sky_components=[
            {
                "kind": "point",
                "ra_dec": phase_center[0],
                "flux": np.outer(total_flux_density(freqs), [1.0, 0.0, 0.0, 1.0]),
            }
        ],
        noise_params={
            "t_receiver": row.t_sys_K,
            "t_atmos": 0.0,
            "t_cmb": 0.0,
            "ant_efficiency": row.ant_efficiency,
            "corr_efficiency": ETA_C,
            "quantization_efficiency": ETA_Q,
            "random_seed": band,
        },
        overwrite=True,
        report="notebook" if band == 3 else None,
    )

## Image with the cube imager

`image_cube_single_field` makes a 128 × 128 Stokes I and Q cube of every processing set with natural weighting,
cleans it with Hogbom down to three times the expected channel noise and restores it. The recovered flux density is
the restored Stokes I at the phase centre, and its error the robust rms (1.4826 × MAD) of the residual channel. The
weighted mean of the real part of the visibilities, the maximum-likelihood flux density of a point source at the
phase centre in the simulated data, is kept to separate the imaging from the noise.
The report of the Band 3 cube is shown below the cell.

In [ ]:
B_MAX, IMAGE_SIZE = 500.0, 128  # longest C-3 baseline (m), pixels
rows = []
for band in bands.index:
    freqs, _ = channel_frequencies(band)
    cell = c / freqs[-1] / B_MAX / 5
    sigma_channel = bands.loc[band, "sigma_uJy"] * 1e-6 * np.sqrt(N_CHAN)
    image_cube_single_field(
        ps_store=store(band, "ps"),
        image_store=store(band, "img"),
        image_params={
            "image_size": [IMAGE_SIZE, IMAGE_SIZE],
            "cell_size": np.array([-cell, cell]),
            "phase_direction": phase_center[0],
            "frequency_coords": freqs,
            "polarization_coords": ["I", "Q"],
            "time_coords": [0],
            "fft_padding": 1.2,
        },
        imaging_weights_params={"weighting": "natural"},
        cycle_control_params={
            "max_iter": 1000,
            "max_cycles": -1,
            "threshold": 3 * sigma_channel,
            "gain": 0.1,
            "psf_sidelobe_factor": 1.5,
            "min_psf_fraction": 0.05,
            "max_psf_fraction": 0.8,
            "max_iter_per_cycle": -1,
        },
        deconvolver="hogbom",
        scan_intents="OBSERVE_TARGET#ON_SOURCE",
        image_data_variables_keep=[
            "sky_residual",
            "sky_model",
            "point_spread_function",
            "primary_beam",
            "beam_fit_params_point_spread_function",
        ],
        processing_set_data_group_name="base",
        single_precision_image=False,
        overwrite=True,
        restore=True,
        report="notebook" if band == 3 else None,
    )
    img = load_image(store(band, "img")).isel(time=0, polarization=0)
    residual = img.SKY_RESIDUAL.values
    rms = 1.4826 * np.median(
        np.abs(residual - np.median(residual, axis=(1, 2), keepdims=True)), axis=(1, 2)
    )
    restored = img.SKY_RESTORED.values[:, IMAGE_SIZE // 2, IMAGE_SIZE // 2]
    ms = next(iter(open_processing_set(store(band, "ps")).values())).ds
    weight = ms.WEIGHT.values * ~ms.FLAG.values
    vis_mean = (ms.VISIBILITY.values.real * weight).sum(axis=(0, 1, 3)) / weight.sum(
        axis=(0, 1, 3)
    )
    model = total_flux_density(freqs)
    for f, m, value, vis, noise in zip(
        freqs, model, restored, vis_mean, rms, strict=True
    ):
        rows.append(
            {
                "band": band,
                "freq_GHz": f / 1e9,
                "model_mJy": 1e3 * m,
                "recovered_mJy": 1e3 * value,
                "visibility_mJy": 1e3 * vis,
                "rms_mJy": 1e3 * noise,
                "expected_rms_mJy": 1e3 * sigma_channel,
            }
        )
results = pd.DataFrame(rows)
results["pull"] = (results.recovered_mJy - results.model_mJy) / results.rms_mJy

## Recovered spectrum

Band means of the four channels (circles, with $\pm 1\sigma$ error bars inside the markers) on the simulated total
(black) and its components (colours), and below, the difference of every channel from the total in units of the
noise.

In [ ]:
COLOR = {
    "dust": "#e34948",
    "free-free": "#2a78d6",
    "synchrotron": "#1baf7a",
    "AME": "#eda100",
}
band_mean = results.groupby("band").agg(
    freq_GHz=("freq_GHz", "mean"),
    model_mJy=("model_mJy", "mean"),
    recovered_mJy=("recovered_mJy", "mean"),
    rms_mJy=("rms_mJy", lambda s: np.sqrt(np.sum(s**2)) / len(s)),
)
band_mean["pull"] = (band_mean.recovered_mJy - band_mean.model_mJy) / band_mean.rms_mJy

nu = np.geomspace(33e9, 1000e9, 500)
fig, (ax, ax_pull) = plt.subplots(
    2, 1, figsize=(8, 7), sharex=True, height_ratios=[3, 1], layout="constrained"
)
for component in LUMINOSITY:
    ax.plot(
        nu / 1e9,
        1e3 * flux_density(component, nu),
        color=COLOR[component],
        lw=1.2,
        label=component,
    )
ax.plot(
    nu / 1e9, 1e3 * total_flux_density(nu), color="0.1", lw=2, label="total, simulated"
)
ax.errorbar(
    band_mean.freq_GHz,
    band_mean.recovered_mJy,
    band_mean.rms_mJy,
    fmt="o",
    ms=7,
    mfc="white",
    mec="0.1",
    mew=1.5,
    ecolor="0.1",
    label="total, recovered (band mean)",
)
ax.set(xscale="log", yscale="log", ylim=(0.1, 3e4), ylabel="flux density (mJy)")
ax.set_title(
    "Single field, ALMA 12 m array, 1 h per band, z = 0.01 starburst",
    loc="left",
    fontsize=11,
)
ax.legend(loc="upper left", frameon=False)
top = ax.secondary_xaxis("top")
top.set_xticks(bands.freq_GHz, labels=bands.index.astype(str))
top.xaxis.set_minor_locator(mticker.NullLocator())
top.set_xlabel("ALMA band")
ax_pull.axhspan(-2, 2, color="0.94", lw=0)
ax_pull.axhline(0, color="0.6", lw=1)
ax_pull.plot(results.freq_GHz, results.pull, "o", ms=5, mfc="white", mec="0.1", mew=1.2)
ax_pull.set(ylim=(-4, 4), ylabel="(recovered − total) / σ", xlabel="frequency (GHz)")
ax_pull.set_xticks([40, 60, 100, 200, 400, 600, 1000])
ax_pull.xaxis.set_major_formatter(mticker.ScalarFormatter())
ax_pull.xaxis.set_minor_formatter(mticker.NullFormatter())
for axis in (ax, ax_pull):
    axis.grid(True, which="major", color="0.9", lw=0.6)
    axis.set_axisbelow(True)

Band means in mJy, and the difference in units of the noise:

In [ ]:
band_mean.round(
    {"freq_GHz": 2, "model_mJy": 3, "recovered_mJy": 3, "rms_mJy": 4, "pull": 2}
)

In [ ]:
noise = results.groupby("band")[["rms_mJy", "expected_rms_mJy"]].mean()
noise["measured / calculator"] = noise.rms_mJy / noise.expected_rms_mJy
mean_squared_pull = np.mean(results.pull**2)
imaging = np.abs(results.recovered_mJy - results.visibility_mJy) / results.rms_mJy
print(f"mean squared pull: {mean_squared_pull:.2f} over {len(results)} channels")
print(f"restored minus visibility mean: at most {imaging.max():.2f} sigma")
display(noise.round(4))
assert mean_squared_pull < 2
assert imaging.max() < 0.25
assert np.allclose(noise["measured / calculator"], 1, atol=0.08)

## Result

The total SED is recovered in every band. The restored flux densities scatter about the simulated total with a mean
squared pull per channel of 0.71 over 40 channels (pure noise gives 1 ± 0.22, and 0.71 or less in 9 % of draws). The
restored value at the phase centre equals the weighted mean of the visibilities to 0.07σ at most, so the imaging adds
nothing measurable to the noise, and the measured noise matches the Sensitivity Calculator's within 3.3 % in every
band. The largest component of the total is the free-free in Band 1, the AME of the PDR (at the paper's deliberately
high column; it peaks at 161 GHz) in Bands 2 and 3, and the dust from Band 4 up.

## Clean up

In [ ]:
import glob
import shutil

for path in (
    glob.glob("sed_B*.ps.zarr")
    + glob.glob("sed_B*.img.zarr")
    + glob.glob("sed_B*_report_files")
):
    shutil.rmtree(path, ignore_errors=True)
viper_client.close()